# 第一讲 从纸笔选择到可重跑的预测

先读实验指南并保留自己的纸笔答案。数据全为合成，公开标签不构成保密盲测。第一次从上到下运行，每阶段先预测输出。第 3、4 讲会完整解释 Python 语法与函数。

## 1 查看训练记录
这一格只导入计算工具、读取训练数据，不读测试标签。这一格会核对当前目录是否含 experiment.py 与 data 文件夹。

In [1]:
from pathlib import Path
import json
import experiment as exp

DATA = Path("data")
OUT = Path("notebook_results")
training = exp.read_records(DATA / "train.csv")
print("实验文件检查：", Path("experiment.py").is_file() and DATA.is_dir())
for row in training:
    print(row)

实验文件检查： True
{'id': 'H1', 'area_m2': 50.0, 'price_wan': 110.0}
{'id': 'H2', 'area_m2': 60.0, 'price_wan': 130.0}
{'id': 'H3', 'area_m2': 80.0, 'price_wan': 170.0}
{'id': 'H4', 'area_m2': 90.0, 'price_wan': 190.0}


## 2 重复一行手算
C 对 50 平方米的预测是 1.5 × 50 + 40。运行前在纸上写出预测和绝对误差。

In [2]:
area = 50
label = 110
prediction = exp.predict(area, 1.5, 40)
print("预测：", prediction)
print("预测减标签：", prediction-label)
print("绝对误差：", abs(prediction-label))
assert prediction == 115

预测： 115.0
预测减标签： 5.0
绝对误差： 5.0


## 3 比较训练误差并冻结
只用四行训练记录选候选，同时固定训练价格均值基线。

In [3]:
model = exp.train(DATA, OUT)
print(json.dumps(model, ensure_ascii=False, indent=2))
assert model["candidate"] == "B"
assert model["training_mae_wan"] == {"A": 10.0, "B": 0.0, "C": 7.5}
assert model["baseline_price_wan"] == 150.0

{
  "candidate": "B",
  "w": 2.0,
  "b": 10.0,
  "training_mae_wan": {
    "A": 10.0,
    "B": 0.0,
    "C": 7.5
  },
  "baseline_price_wan": 150.0,
  "training_sha256": "2dff3f62e1ef5229b9c49930c1a73853704b3b3918562fb89fb9beab027929c7"
}


## 4 先留下测试预测
这一步不读标签。运行前手算 65 与 85 的两个预测，再核对。

In [4]:
predictions = exp.make_predictions(DATA, OUT)
print(json.dumps(predictions, ensure_ascii=False, indent=2))
assert [r["prediction_wan"] for r in predictions] == [140.0, 180.0]

[
  {
    "id": "T1",
    "prediction_wan": 140.0,
    "baseline_wan": 150.0
  },
  {
    "id": "T2",
    "prediction_wan": 180.0,
    "baseline_wan": 150.0
  }
]


## 5 现在才打开答案
若你已从正文看过答案，诚实记录这是流程复现。评价读取冻结记录与已保存预测，不重新选择参数。

In [5]:
metrics = exp.evaluate(DATA, OUT)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
assert metrics["test_mae_wan"] == 5.0
assert metrics["baseline_test_mae_wan"] == 15.0

{
  "selected_candidate": "B",
  "test_predictions_wan": [
    140.0,
    180.0
  ],
  "test_labels_wan": [
    144.0,
    174.0
  ],
  "absolute_errors_wan": [
    4.0,
    6.0
  ],
  "test_mae_wan": 5.0,
  "baseline_test_mae_wan": 15.0,
  "scope": "两条公开的合成教学记录，不是真实房价能力估计"
}


## 6 为什么误差不能直接抵消
预测 90 与 110，标签都是 100。两次误差的方向相反，但都偏 10。

In [6]:
signed_mean = ((90-100)+(110-100))/2
absolute_mean = exp.mae([90,110],[100,100])
print("带方向平均：", signed_mean)
print("平均绝对误差：", absolute_mean)
assert signed_mean == 0
assert absolute_mean == 10

带方向平均： 0.0
平均绝对误差： 10.0


## 7 训练零误差不够
以下 M 只记住训练答案，对其他面积返回 999。这是一个逻辑反例，不是推荐模型。

In [7]:
memory = {row["area_m2"]: row["price_wan"] for row in training}
train_predictions = [memory.get(row["area_m2"],999) for row in training]
new_predictions = [memory.get(area,999) for area in [65,85]]
print("M 训练 MAE：", exp.mae(train_predictions,[row["price_wan"] for row in training]))
print("M 教学测试 MAE：", exp.mae(new_predictions,[144,174]))
assert exp.mae(new_predictions,[144,174]) == 840

M 训练 MAE： 0.0
M 教学测试 MAE： 840.0


## 8 运行边界检查
这些检查验证若干明确要求；通过不等于证明代码没有任何错误，也不等于数据有现实代表性。

In [8]:
exp.self_test()

SELF_TEST_PASS: 9 explicit checks


## 9 写下你的结论
在新文字单元中回答：训练、预测、评价分别读取了哪些文件？你能确认哪些数？为什么不能声称已经能给真实房屋估价？最后重启内核并运行全部单元，确认不依赖未记录的历史状态。